# Session 09 — UDFs & Schema Management
**Topics:** Define UDF · Apply UDF · Why UDFs are slow · StructType / StructField · Read with explicit schema

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, BooleanType, DateType, LongType
)
from pyspark.sql.functions import udf
import time

try:
    spark
except NameError:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.appName("Session09").master("local[*]").getOrCreate()
    spark.sparkContext.setLogLevel("ERROR")

---
## 1 · Define and Register a UDF

A **UDF (User Defined Function)** lets you apply custom Python logic on a DataFrame column.  
You write a plain Python function, then wrap it with `udf()` and declare the return type.

```python
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def my_func(value):
    return ...  # some Python logic

my_udf = udf(my_func, StringType())   # wrap + declare return type
```

In [0]:
data = [
    (1, "Alice",   85000, "IN"),
    (2, "Bob",     42000, "US"),
    (3, "Charlie", 61000, "UK"),
    (4, "Diana",   95000, "IN"),
    (5, "Eve",     38000, "US"),
    (6, "Frank",   72000, "UK"),
]
emp_df = spark.createDataFrame(data, ["id", "name", "salary", "country"])
emp_df.show()

In [0]:
# UDF 1: salary band classifier
def salary_band(salary):
    if salary is None:
        return "Unknown"
    if salary >= 80000:
        return "Band A"
    elif salary >= 60000:
        return "Band B"
    elif salary >= 40000:
        return "Band C"
    else:
        return "Band D"

salary_band_udf = udf(salary_band, StringType())

In [0]:
# Apply UDF on the salary column
emp_df.withColumn("band", salary_band_udf(F.col("salary"))).show()

In [0]:
# UDF 2: using the @udf decorator syntax (cleaner for simple functions)
@udf(StringType())
def country_name(code):
    mapping = {"IN": "India", "US": "United States", "UK": "United Kingdom"}
    return mapping.get(code, "Unknown")

emp_df.withColumn("country_full", country_name(F.col("country"))).show()

In [0]:
# UDF 3: UDF that takes multiple columns as input
@udf(StringType())
def label(name, salary, country):
    band = "Senior" if salary >= 70000 else "Junior"
    return f"{name} | {band} | {country}"

emp_df.withColumn("summary", label("name", "salary", "country")).show(truncate=False)

---
## 2 · Why UDFs Are Slow — When to Avoid Them

Spark is built on the JVM. When a UDF runs, each row is:
1. Serialised from JVM → Python
2. Processed in Python
3. Serialised back Python → JVM

This per-row serialisation overhead makes UDFs significantly slower than built-in functions, which execute entirely inside the JVM and benefit from the Catalyst optimizer.

**Rule of thumb:** If a built-in `F.*` function can do the job — use it. Reserve UDFs for logic that genuinely cannot be expressed with built-ins.

In [0]:
# Side-by-side: UDF vs built-in for the same transformation

# ── Using UDF ────────────────────────────────────────────────────────────────
@udf(StringType())
def upper_udf(s):
    return s.upper() if s else s

# ── Using built-in ───────────────────────────────────────────────────────────
emp_df.select(
    "name",
    upper_udf(F.col("name")).alias("udf_result"),
    F.upper(F.col("name")).alias("builtin_result")
).show()

In [0]:
# Common cases where built-ins replace UDFs:

emp_df.select(
    # Instead of a UDF for conditional logic → use when().otherwise()
    F.when(F.col("salary") >= 80000, "Band A")
     .when(F.col("salary") >= 60000, "Band B")
     .otherwise("Band C").alias("band_builtin"),

    # Instead of a UDF for null handling → use coalesce() or fillna()
    F.coalesce(F.col("country"), F.lit("Unknown")).alias("country_safe"),

    # Instead of a UDF for string manipulation → use built-in string functions
    F.concat(F.col("name"), F.lit(" ("), F.col("country"), F.lit(")")).alias("name_country")
).show(truncate=False)

---
## 3 · StructType and StructField

```python
schema = StructType([
    StructField("column_name", DataType(), nullable=True/False),
    ...
])
```

- `StructType`  → the whole schema (list of fields)
- `StructField` → one column: name, type, nullable flag
- `nullable=False` → Spark will not accept null values in that column

In [0]:
# Define a schema explicitly
employee_schema = StructType([
    StructField("emp_id",     IntegerType(), nullable=False),
    StructField("name",       StringType(),  nullable=False),
    StructField("department", StringType(),  nullable=True),
    StructField("salary",     DoubleType(),  nullable=True),
    StructField("is_active",  BooleanType(), nullable=True),
])

rows = [
    (101, "Alice",   "Engineering", 85000.0, True),
    (102, "Bob",     "Marketing",   42000.0, True),
    (103, "Charlie", None,          61000.0, False),
    (104, "Diana",   "Engineering", None,    True),
]

schema_df = spark.createDataFrame(rows, schema=employee_schema)
schema_df.printSchema()

In [0]:
schema_df.show()

In [0]:
# Nested schema — StructField can contain another StructType
address_schema = StructType([
    StructField("emp_id", IntegerType(), nullable=False),
    StructField("name",   StringType(),  nullable=False),
    StructField("address", StructType([
        StructField("city",    StringType(), nullable=True),
        StructField("pincode", StringType(), nullable=True),
    ]), nullable=True),
])

from pyspark.sql import Row
nested_rows = [
    Row(emp_id=1, name="Alice",   address=Row(city="Chennai",   pincode="600001")),
    Row(emp_id=2, name="Bob",     address=Row(city="Mumbai",    pincode="400001")),
    Row(emp_id=3, name="Charlie", address=None),
]
nested_df = spark.createDataFrame(nested_rows, schema=address_schema)
nested_df.printSchema()

In [0]:
# Accessing nested fields with dot notation
nested_df.select(
    "emp_id",
    "name",
    F.col("address.city").alias("city"),
    F.col("address.pincode").alias("pincode")
).show()

---
## 4 · Read Files with an Explicit Schema

By default `spark.read` with `inferSchema=True` scans the file to guess types — it works but:
- Requires an extra pass over the data (slow on large files)
- Can infer wrong types (e.g. `"00123"` inferred as Integer, leading zeros lost)
- Nullable is always `true` for every column

Providing a schema explicitly avoids all of this.

In [0]:
# ── Inferred schema (default) ─────────────────────────────────────────────
path = "/Volumes/workspace/default/demo_volume"
csv_file = f"{path}/employee_file.csv"
json_file = f"{path}/order_file.json"

inferred_df = spark.read.option("header", True).option("inferSchema", True).csv(csv_file)
print("=== Inferred Schema ===")
inferred_df.printSchema()

In [0]:
# ── Explicit schema ────────────────────────────────────────────────────────
emp_schema = StructType([
    StructField("emp_id",     IntegerType(), nullable=False),
    StructField("name",       StringType(),  nullable=False),
    StructField("department", StringType(),  nullable=True),
    StructField("salary",     DoubleType(),  nullable=True),
    StructField("join_date",  DateType(),    nullable=True),
])

explicit_df = spark.read \
    .option("header", True) \
    .option("dateFormat", "yyyy-MM-dd") \
    .schema(emp_schema) \
    .csv(csv_file)

print("=== Explicit Schema ===")
explicit_df.printSchema()

In [0]:
explicit_df.show()

In [0]:
# Same pattern works for JSON
order_schema = StructType([
    StructField("order_id",   IntegerType(), nullable=False),
    StructField("customer",   StringType(),  nullable=False),
    StructField("amount",     DoubleType(),  nullable=True),
    StructField("order_date", DateType(),    nullable=True),
])

orders_df = spark.read \
    .option("dateFormat", "yyyy-MM-dd") \
    .schema(order_schema) \
    .json(json_file)

orders_df.printSchema()
orders_df.show()

In [0]:
# What happens when data doesn't match the schema?
# Spark sets mismatched values to null instead of crashing

wrong_schema = StructType([
    StructField("emp_id",     IntegerType(), nullable=True),
    StructField("name",       StringType(),  nullable=True),
    StructField("department", IntegerType(), nullable=True),  # wrong type — should be String
    StructField("salary",     DoubleType(),  nullable=True),
    StructField("join_date",  DateType(),    nullable=True),
])

mismatch_df = spark.read.option("header", True).schema(wrong_schema).csv(csv_file)
print("department column with wrong type (IntegerType) → all nulls:")
mismatch_df.select("name", "department").show()

---
## Quick Reference

**UDF**
```python
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

my_udf = udf(lambda x: x.upper(), StringType())          # inline

@udf(StringType())                                        # decorator
def my_udf(x): return x.upper()

df.withColumn("new_col", my_udf(F.col("col")))
```

**Schema**
```python
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField("id",   IntegerType(), nullable=False),
    StructField("name", StringType(),  nullable=True),
])
df = spark.read.schema(schema).csv("path/to/file.csv")
```

**When to avoid UDFs**

| Scenario | Use instead |
|---|---|
| if/elif/else logic | `F.when().otherwise()` |
| String manipulation | `F.upper()` `F.concat()` `F.regexp_replace()` |
| Null handling | `F.coalesce()` `F.fillna()` |
| Math operations | `F.round()` `F.abs()` `F.pow()` |
| Date handling | `F.to_date()` `F.datediff()` `F.date_format()` |